## Training set exploration

### Columns description:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('./data/application_train.csv')
print(df.shape)
print(df.columns.tolist())

(307511, 122)
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONA

### Target count

In [3]:
print(df['TARGET'].value_counts())
print(df['TARGET'].value_counts(normalize=True).round(3))

TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


### Missing values

In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_df)

                          missing_count  missing_pct
COMMONAREA_MEDI                  214865         69.9
COMMONAREA_AVG                   214865         69.9
COMMONAREA_MODE                  214865         69.9
NONLIVINGAPARTMENTS_MEDI         213514         69.4
NONLIVINGAPARTMENTS_MODE         213514         69.4
...                                 ...          ...
EXT_SOURCE_2                        660          0.2
AMT_GOODS_PRICE                     278          0.1
DAYS_LAST_PHONE_CHANGE                1          0.0
CNT_FAM_MEMBERS                       2          0.0
AMT_ANNUITY                          12          0.0

[67 rows x 2 columns]


In [5]:
print(missing_df[missing_df['missing_pct'] > 40])

                              missing_count  missing_pct
COMMONAREA_MEDI                      214865         69.9
COMMONAREA_AVG                       214865         69.9
COMMONAREA_MODE                      214865         69.9
NONLIVINGAPARTMENTS_MEDI             213514         69.4
NONLIVINGAPARTMENTS_MODE             213514         69.4
NONLIVINGAPARTMENTS_AVG              213514         69.4
LIVINGAPARTMENTS_MODE                210199         68.4
LIVINGAPARTMENTS_MEDI                210199         68.4
LIVINGAPARTMENTS_AVG                 210199         68.4
FONDKAPREMONT_MODE                   210295         68.4
FLOORSMIN_MODE                       208642         67.8
FLOORSMIN_MEDI                       208642         67.8
FLOORSMIN_AVG                        208642         67.8
YEARS_BUILD_MODE                     204488         66.5
YEARS_BUILD_MEDI                     204488         66.5
YEARS_BUILD_AVG                      204488         66.5
OWN_CAR_AGE                    

In [7]:
desc = pd.read_csv('./data/HomeCredit_columns_description.csv', encoding='latin-1')
cols_high_missing = missing_df[missing_df['missing_pct'] > 40].index.tolist()
print(desc[desc['Row'].isin(cols_high_missing)][['Row', 'Description']])

                             Row  \
21                   OWN_CAR_AGE   
41                  EXT_SOURCE_1   
44                APARTMENTS_AVG   
45              BASEMENTAREA_AVG   
46   YEARS_BEGINEXPLUATATION_AVG   
47               YEARS_BUILD_AVG   
48                COMMONAREA_AVG   
49                 ELEVATORS_AVG   
50                 ENTRANCES_AVG   
51                 FLOORSMAX_AVG   
52                 FLOORSMIN_AVG   
53                  LANDAREA_AVG   
54          LIVINGAPARTMENTS_AVG   
55                LIVINGAREA_AVG   
56       NONLIVINGAPARTMENTS_AVG   
57             NONLIVINGAREA_AVG   
58               APARTMENTS_MODE   
59             BASEMENTAREA_MODE   
60  YEARS_BEGINEXPLUATATION_MODE   
61              YEARS_BUILD_MODE   
62               COMMONAREA_MODE   
63                ELEVATORS_MODE   
64                ENTRANCES_MODE   
65                FLOORSMAX_MODE   
66                FLOORSMIN_MODE   
67                 LANDAREA_MODE   
68         LIVINGAPARTMENTS_

We are removing columns with more than 50% missing value except External source that is a critical feature as it is a score given by an external agency. 

In [8]:
threshold = 40
cols_to_drop = missing_df[missing_df['missing_pct'] > threshold].index.tolist()

# Garder EXT_SOURCE_1
cols_to_drop.remove('EXT_SOURCE_1')

print(f"Colonnes droppées : {len(cols_to_drop)}")
df = df.drop(columns=cols_to_drop)
print(f"Shape après drop : {df.shape}")

Colonnes droppées : 48
Shape après drop : (307511, 74)


Remaining missing:

In [9]:
missing2 = df.isnull().sum()
missing_pct2 = (missing2 / len(df) * 100).round(1)
missing_df2 = pd.DataFrame({'missing_count': missing2, 'missing_pct': missing_pct2})
missing_df2 = missing_df2[missing_df2['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_df2)

                            missing_count  missing_pct
EXT_SOURCE_1                       173378         56.4
OCCUPATION_TYPE                     96391         31.3
EXT_SOURCE_3                        60965         19.8
AMT_REQ_CREDIT_BUREAU_YEAR          41519         13.5
AMT_REQ_CREDIT_BUREAU_QRT           41519         13.5
AMT_REQ_CREDIT_BUREAU_MON           41519         13.5
AMT_REQ_CREDIT_BUREAU_WEEK          41519         13.5
AMT_REQ_CREDIT_BUREAU_DAY           41519         13.5
AMT_REQ_CREDIT_BUREAU_HOUR          41519         13.5
NAME_TYPE_SUITE                      1292          0.4
DEF_60_CNT_SOCIAL_CIRCLE             1021          0.3
DEF_30_CNT_SOCIAL_CIRCLE             1021          0.3
OBS_60_CNT_SOCIAL_CIRCLE             1021          0.3
OBS_30_CNT_SOCIAL_CIRCLE             1021          0.3
EXT_SOURCE_2                          660          0.2
AMT_GOODS_PRICE                       278          0.1
DAYS_LAST_PHONE_CHANGE                  1          0.0
CNT_FAM_ME

Replacing missing values: 
categories: "unknown"
numerical: median

In [12]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')

print(f"Missing restants : {df.isnull().sum().sum()}")

Missing restants : 0


Let's create derived features: 

In [13]:
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

print(df[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'EMPLOYED_TO_AGE_RATIO', 'CREDIT_TERM', 'INCOME_PER_PERSON']].describe())

       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  EMPLOYED_TO_AGE_RATIO  \
count        307511.000000         307511.000000          307511.000000   
mean              3.957570              0.180929              -2.920135   
std               2.689728              0.094573               6.627098   
min               0.004808              0.000224             -47.489663   
25%               2.018667              0.114782               0.021559   
50%               3.265067              0.162833               0.088645   
75%               5.159880              0.229067               0.191000   
max              84.736842              1.875965               0.728811   

         CREDIT_TERM  INCOME_PER_PERSON  
count  307511.000000       3.075110e+05  
mean        0.053695       9.310634e+04  
std         0.022482       1.013733e+05  
min         0.016790       2.812500e+03  
25%         0.036900       4.725000e+04  
50%         0.050000       7.500000e+04  
75%         0.064043       1.1

In [14]:
print(df['DAYS_EMPLOYED'].describe())
print("---")
print(df['DAYS_BIRTH'].describe())

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64
---
count    307511.000000
mean     -16036.995067
std        4363.988632
min      -25229.000000
25%      -19682.000000
50%      -15750.000000
75%      -12413.000000
max       -7489.000000
Name: DAYS_BIRTH, dtype: float64


In [15]:
print((df['DAYS_EMPLOYED'] == 365243).sum())


55374


In [16]:
# Remplacer 365243 par NaN puis imputer par la médiane
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].fillna(df['DAYS_EMPLOYED'].median())

# Recalculer les features dérivées proprement
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']

print(df['EMPLOYED_TO_AGE_RATIO'].describe())

count    307511.000000
mean          0.142371
std           0.124883
min          -0.000000
25%           0.066852
50%           0.091037
75%           0.191054
max           0.728811
Name: EMPLOYED_TO_AGE_RATIO, dtype: float64


In [17]:
df.groupby('TARGET')[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 
                       'EMPLOYED_TO_AGE_RATIO', 'AMT_INCOME_TOTAL', 
                       'DAYS_BIRTH']].mean()

,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,EMPLOYED_TO_AGE_RATIO,AMT_INCOME_TOTAL,DAYS_BIRTH
TARGET,,,,,
0,3.963729,0.180529,0.144207,169077.722266,-16138.176397
1,3.887438,0.185482,0.121468,165611.760906,-14884.828077


In [18]:
corr = df.corr(numeric_only=True)['TARGET'].sort_values()
print(corr.head(10))
print("---")
print(corr.tail(10))

EXT_SOURCE_2                 -0.160295
EXT_SOURCE_3                 -0.155892
EXT_SOURCE_1                 -0.098887
EMPLOYED_TO_AGE_RATIO        -0.049603
AMT_GOODS_PRICE              -0.039623
REGION_POPULATION_RELATIVE   -0.037227
AMT_CREDIT                   -0.030369
FLAG_DOCUMENT_6              -0.028602
HOUR_APPR_PROCESS_START      -0.024166
FLAG_PHONE                   -0.023806
Name: TARGET, dtype: float64
---
REG_CITY_NOT_LIVE_CITY         0.044395
FLAG_EMP_PHONE                 0.045982
REG_CITY_NOT_WORK_CITY         0.050994
DAYS_ID_PUBLISH                0.051457
DAYS_LAST_PHONE_CHANGE         0.055218
REGION_RATING_CLIENT           0.058899
REGION_RATING_CLIENT_W_CITY    0.060893
DAYS_EMPLOYED                  0.063368
DAYS_BIRTH                     0.078239
TARGET                         1.000000
Name: TARGET, dtype: float64


In [19]:
df.groupby('TARGET')[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean()

,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
TARGET,,,
0,0.508396,0.52357,0.523767
1,0.457791,0.41126,0.423775


In [20]:
print(df.select_dtypes(include=['object']).nunique())

NAME_CONTRACT_TYPE             2
CODE_GENDER                    3
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
NAME_TYPE_SUITE                8
NAME_INCOME_TYPE               8
NAME_EDUCATION_TYPE            5
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
OCCUPATION_TYPE               19
WEEKDAY_APPR_PROCESS_START     7
ORGANIZATION_TYPE             58
dtype: int64


In [22]:
print(df['CODE_GENDER'].value_counts())
print(df[df['CODE_GENDER'] == 'XNA']['TARGET'].value_counts())


CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64
TARGET
0    4
Name: count, dtype: int64


In [23]:
df = df[df['CODE_GENDER'] != 'XNA']
print(df.shape)

(307507, 79)


In [24]:
cols_to_check = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
for col in cols_to_check:
    q99 = df[col].quantile(0.99)
    q01 = df[col].quantile(0.01)
    print(f"{col} — 1%: {q01:.0f} | 99%: {q99:.0f} | max: {df[col].max():.0f} | min: {df[col].min():.0f}")

AMT_INCOME_TOTAL — 1%: 45000 | 99%: 472500 | max: 117000000 | min: 25650
AMT_CREDIT — 1%: 76410 | 99%: 1854000 | max: 4050000 | min: 45000
AMT_ANNUITY — 1%: 6183 | 99%: 70006 | max: 258026 | min: 1616
DAYS_BIRTH — 1%: -24419 | 99%: -8263 | max: -7489 | min: -25229
DAYS_EMPLOYED — 1%: -10895 | 99%: -116 | max: 0 | min: -17912


Outlier with 117M $ income will derive XGBoost prediction -> caping 99 percentile. 

In [25]:
for col in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']:
    q99 = df[col].quantile(0.99)
    q01 = df[col].quantile(0.01)
    df[col] = df[col].clip(lower=q01, upper=q99)

print(df[['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']].describe())

       AMT_INCOME_TOTAL    AMT_CREDIT    AMT_ANNUITY
count     307507.000000  3.075070e+05  307507.000000
mean      166067.210573  5.963086e+05   26945.299318
std        83000.528686  3.913093e+05   13654.603478
min        45000.000000  7.641000e+04    6183.000000
25%       112500.000000  2.700000e+05   16524.000000
50%       147150.000000  5.135310e+05   24903.000000
75%       202500.000000  8.086500e+05   34596.000000
max       472500.000000  1.854000e+06   70006.500000


In [ ]:
df.to_csv('./data/clean/application_train_clean.csv', index=False)
print(f"Dataset sauvegardé : {df.shape}")


Dataset sauvegardé : (307507, 79)
